# Reproduce `chain-9b`

Generated from `<inference page: chain-9b x 22356 instance(s)>`. The notebook replays inference + evaluation against the exact config snapshot that produced the original run, so a re-execution should produce a comparable `model_patch` (LLM determinism caveats notwithstanding).

- **Instances:** 22356
- **Config id:** `chain-9b`


## 1. Setup

The notebook's `kernelspec.name = "evomas"` (see metadata at the bottom of the file) tells Jupyter / VSCode to auto-pick the interpreter `setup.ps1` / `setup.sh` registered for `~/.evomas-venv`. As a safety net the first cell also prepends the venv's site-packages to `sys.path` — so even if the kernel falls back to a generic Python 3 (different machine, no `setup.ps1` run), the evomas imports still resolve. Adjust `OLLAMA_BASE_URL` if your Ollama daemon isn't on the default host; `SWEBENCH_API_KEY` is only required by the remote-eval cell at the bottom.

### Picking the kernel in VSCode

If VSCode opens the notebook outside the EvoMas workspace (e.g. straight from `~/Downloads`), it won't auto-resolve the kernelspec and asks you to **Select Kernel**. Two-tier picker:

- **"Python Environments…"** lists raw Python interpreters discovered by the Python extension (system Python, conda envs, `.venv`/`venv` folders inside workspaces). `~/.evomas-venv` is outside the conventional discovery paths, so it does NOT show up here.
- **"Jupyter Kernel…"** lists registered Jupyter kernelspecs (`%APPDATA%\jupyter\kernels\*` on Windows, `~/.local/share/jupyter/kernels/*` on Linux/mac). This is where the EvoMas one lives — pick **"Python 3 (EvoMas)"** here. VSCode remembers the choice per-notebook so you only have to do it once.

If the entry doesn't appear there: `Ctrl+Shift+P` → **"Developer: Reload Window"** so the Jupyter extension re-scans kernelspecs, or run `jupyter kernelspec list` to confirm `evomas` is registered (if not, re-run `setup.ps1` / `setup.sh`).

In [1]:
import os
import sys
import json
import subprocess
from pathlib import Path

# Defensive sys.path prepend so `import evomas...` still
# resolves when the kernel isn't the evomas-venv one.
_venv = Path.home() / '.evomas-venv'
if _venv.is_dir() and str(_venv) not in sys.executable:
    for _sp in (_venv / 'Lib' / 'site-packages',
                _venv / 'lib' / 'site-packages'):
        if _sp.is_dir() and str(_sp) not in sys.path:
            sys.path.insert(0, str(_sp))

import evomas.paths  # noqa: F401  # triggers load_dotenv(evomas/.env)

from evomas.core.workflow.runner import run as run_evomas
from evomas.utils.instances import fetch_swebench_instances

# Mirror logging to a per-run text file so generate_report.py
# can mine the same lines the API matrix path writes.
import logging
RUN_OUTPUT_DIR = Path('notebook-chain-9b').resolve()
RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = RUN_OUTPUT_DIR / 'inference.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    force=True,
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(LOG_FILE, encoding='utf-8'),
    ],
)
print(f'Mirroring inference logs to {LOG_FILE}')


Mirroring inference logs to C:\Users\XF\Desktop\TFG\EvoMas\notebooks\notebook-chain-9b\inference.log


### Environment variables

All run-time configuration the notebook needs lives in this cell — edit values here rather than chasing them through the code. Each assignment **overrides** whatever is in the environment / .env when the cell runs.

- **`OLLAMA_BASE_URL`** — where the Ollama daemon serves.
- **`SWEBENCH_API_KEY`** — required by the remote-eval cell (section 5) when running `--remote` against sb-cli. Local Docker harness runs don't need it.
- **`EVOMAS_INSTANCES`** — override the SWE-bench instance cache location. The next cell already searches sensible defaults; only set this if your cache is somewhere non-standard.
- **`GOOGLE_API_KEY` / `OPENAI_API_KEY`** — only needed if the inlined config picks a Gemini / OpenAI model instead of Ollama.


In [2]:
# Edit values here; each assignment overrides the inherited
# environment / .env. Uncomment the lines you need.
os.environ['OLLAMA_BASE_URL'] = 'http://192.168.1.50:11434'
# os.environ['SWEBENCH_API_KEY'] = 'swb_...'
# os.environ['EVOMAS_INSTANCES'] = '/path/to/swebench_instances.jsonl'
# os.environ['GOOGLE_API_KEY']   = '...'
# os.environ['OPENAI_API_KEY']   = '...'

# Echo the effective values (mask secrets) so you can verify the cell ran.
for _k in ('OLLAMA_BASE_URL', 'SWEBENCH_API_KEY', 'EVOMAS_INSTANCES',
           'GOOGLE_API_KEY', 'OPENAI_API_KEY'):
    _v = os.environ.get(_k, '')
    if not _v:
        print(f'  {_k:<18} <unset>')
    elif _k.endswith('_API_KEY'):
        print(f'  {_k:<18} {_v[:6]}***({len(_v)} chars)')
    else:
        print(f'  {_k:<18} {_v}')


  OLLAMA_BASE_URL    http://192.168.1.50:11434
  SWEBENCH_API_KEY   swb_QM***(56 chars)
  EVOMAS_INSTANCES   <unset>
  GOOGLE_API_KEY     AIzaSy***(39 chars)
  OPENAI_API_KEY     <unset>


## 2. Inlined config

Exact resolved config the original run used. Tweak hyperparameters here if you want to experiment with variations.

The cell below the config dict renders a mermaid diagram of the topology so you can see the agent-graph shape at a glance. The diagram is regenerated from `CONFIG['edges']` + `CONFIG['agents']` every time the cell runs, so edits to the dict above are reflected immediately.

In [3]:
CONFIG = {   'id': 'chain-9b',
    'description': 'Type-driven linear chain: locator → patcher → reviewer → finalizer. Each agent '
                   'inherits prompts/tools from its type class under evomas/agents/types/ — no '
                   'bespoke Python.',
    'entry': 'locator',
    'end': ['finalizer'],
    'edges': [   {'from': 'locator', 'to': 'patcher'},
                 {'from': 'patcher', 'to': 'reviewer'},
                 {'from': 'reviewer', 'to': 'finalizer'}],
    'agents': {   'locator': {   'class': 'LocatorAgent',
                                 'model': 'ollama/qwen3.5:9b',
                                 'think': False,
                                 'num_ctx': 8192,
                                 'stream': True,
                                 'temperature': 0.2,
                                 'top_k': 40,
                                 'top_p': 0.9,
                                 'min_p': 0,
                                 'repeat_penalty': 1.1,
                                 'repeat_last_n': 64,
                                 'seed': 0,
                                 'num_predict': 512,
                                 'stop': ['</files>'],
                                 'max_iters': 6},
                  'patcher': {   'class': 'PatcherAgent',
                                 'model': 'ollama/qwen3.5:9b',
                                 'think': True,
                                 'num_ctx': 16384,
                                 'stream': True,
                                 'temperature': 0,
                                 'top_k': 40,
                                 'top_p': 0.9,
                                 'min_p': 0,
                                 'repeat_penalty': 1.1,
                                 'repeat_last_n': 64,
                                 'seed': 0,
                                 'num_predict': 2048,
                                 'stop': ['</patch>'],
                                 'max_iters': 12,
                                 'fallback': {'enabled': True, 'guarantee_change': True}},
                  'reviewer': {   'class': 'ReviewerAgent',
                                  'model': 'ollama/qwen3.5:9b',
                                  'think': True,
                                  'num_ctx': 4096,
                                  'stream': True,
                                  'temperature': 0,
                                  'top_k': 40,
                                  'top_p': 0.9,
                                  'min_p': 0,
                                  'repeat_penalty': 1.1,
                                  'repeat_last_n': 64,
                                  'seed': 0,
                                  'num_predict': 1024,
                                  'stop': ['</review>'],
                                  'max_iters': 6},
                  'finalizer': {   'class': 'HelperProxyAgent',
                                   'model': 'ollama/qwen3.5:9b',
                                   'think': True,
                                   'num_ctx': 4096,
                                   'stream': True,
                                   'temperature': 0,
                                   'top_k': 40,
                                   'top_p': 0.9,
                                   'min_p': 0,
                                   'repeat_penalty': 1.1,
                                   'repeat_last_n': 64,
                                   'seed': 0,
                                   'num_predict': 512,
                                   'stop': [],
                                   'max_iters': 4}}}

In [4]:
from IPython.display import Markdown, display

def _topology_mermaid(cfg):
    """Render the topology as a Mermaid flowchart.

    Mirrors what the topology page's cytoscape canvas shows:
    virtual START/END boundary nodes, one node per agent with
    its class as a second-line label, edges directed left-to-
    right. Renders inline in Jupyter Lab + VSCode Jupyter; if
    the cell falls back to plain text the source stays readable.
    """
    lines = ['graph LR']
    lines.append('    START((START))')
    lines.append('    END((END))')
    for name, block in (cfg.get('agents') or {}).items():
        cls = (block or {}).get('class', '') or ''
        label = f'{name}<br/><i>{cls}</i>' if cls else name
        # Backticks break the mermaid parser; strip them.
        label = label.replace('`', '')
        lines.append(f'    {name}["{label}"]')
    entry = cfg.get('entry') or ''
    if entry:
        lines.append(f'    START --> {entry}')
    for e in (cfg.get('edges') or []):
        if isinstance(e, dict) and e.get('from') and e.get('to'):
            lines.append(f'    {e["from"]} --> {e["to"]}')
    end_field = cfg.get('end')
    ends = (
        [end_field] if isinstance(end_field, str) and end_field
        else list(end_field or [])
    )
    # `→ END` only for nodes with no outgoing edges, same
    # rule as `graph_builder.py`.
    out_sources = {e.get('from') for e in (cfg.get('edges') or [])
                   if isinstance(e, dict)}
    for n in ends:
        if n and n not in out_sources:
            lines.append(f'    {n} --> END')
    return '\n'.join(lines)

display(Markdown('```mermaid\n' + _topology_mermaid(CONFIG) + '\n```'))


```mermaid
graph LR
    START((START))
    END((END))
    locator["locator<br/><i>LocatorAgent</i>"]
    patcher["patcher<br/><i>PatcherAgent</i>"]
    reviewer["reviewer<br/><i>ReviewerAgent</i>"]
    finalizer["finalizer<br/><i>HelperProxyAgent</i>"]
    START --> locator
    locator --> patcher
    patcher --> reviewer
    reviewer --> finalizer
    finalizer --> END
```

## 3. Instances

Self-contained: the notebook regenerates its own instances from zero each run. SWE-bench rows get pulled fresh from HuggingFace (cached under `~/.cache/huggingface`); custom rows are reconstructed from the minimal inputs the user added via the Inference page's `+ Custom` modal.

In [5]:
INSTANCE_IDS = [   'astropy__astropy-12907',
    'astropy__astropy-6938',
    'django__django-10914',
    'django__django-10924',
    'django__django-11049',
    'django__django-11133',
    'django__django-11179',
    'django__django-11964',
    'django__django-12113',
    'django__django-12125',
    'django__django-12856',
    'django__django-12908',
    'django__django-13447',
    'django__django-14411',
    'django__django-14534',
    'django__django-14580',
    'django__django-14672',
    'django__django-14915',
    'django__django-15061',
    'django__django-15320',
    'django__django-15388',
    'django__django-15400',
    'django__django-15814',
    'django__django-15851',
    'django__django-15902',
    'django__django-16046',
    'django__django-16255',
    'django__django-16379',
    'django__django-16527',
    'django__django-17087',
    'matplotlib__matplotlib-23314',
    'matplotlib__matplotlib-23476',
    'matplotlib__matplotlib-23563',
    'matplotlib__matplotlib-25433',
    'mwaskom__seaborn-3010',
    'mwaskom__seaborn-3190',
    'mwaskom__seaborn-3407',
    'psf__requests-1963',
    'psf__requests-863',
    'pydata__xarray-4094',
    'pydata__xarray-4248',
    'pydata__xarray-5131',
    'pylint-dev__pylint-7080',
    'pylint-dev__pylint-7993',
    'pytest-dev__pytest-11143',
    'pytest-dev__pytest-11148',
    'pytest-dev__pytest-5227',
    'pytest-dev__pytest-5413',
    'pytest-dev__pytest-6116',
    'pytest-dev__pytest-7168',
    'pytest-dev__pytest-7432',
    'scikit-learn__scikit-learn-13439',
    'scikit-learn__scikit-learn-13584',
    'scikit-learn__scikit-learn-13779',
    'sphinx-doc__sphinx-7738',
    'sphinx-doc__sphinx-8721',
    'sympy__sympy-12171',
    'sympy__sympy-13177',
    'sympy__sympy-13480',
    'sympy__sympy-13647',
    'sympy__sympy-13971',
    'sympy__sympy-14774',
    'sympy__sympy-14817',
    'sympy__sympy-15346',
    'sympy__sympy-15609',
    'sympy__sympy-16988',
    'sympy__sympy-17139',
    'sympy__sympy-17630',
    'sympy__sympy-17655',
    'sympy__sympy-18057',
    'sympy__sympy-18621',
    'sympy__sympy-19487',
    'sympy__sympy-20212',
    'sympy__sympy-21612',
    'sympy__sympy-21614',
    'sympy__sympy-21627',
    'sympy__sympy-22005']


In [6]:
# Pull plan for SWE-bench rows: `{(subset, split): [ids]}`.
# At runtime the cell below calls `fetch_swebench_instances`
# per group and filters down to just these IDs.
SWEBENCH_GROUPS = {   ('lite', 'test'): [   'astropy__astropy-12907',
                          'astropy__astropy-6938',
                          'django__django-10914',
                          'django__django-10924',
                          'django__django-11049',
                          'django__django-11133',
                          'django__django-11179',
                          'django__django-11964',
                          'django__django-12113',
                          'django__django-12125',
                          'django__django-12856',
                          'django__django-12908',
                          'django__django-13447',
                          'django__django-14411',
                          'django__django-14534',
                          'django__django-14580',
                          'django__django-14672',
                          'django__django-14915',
                          'django__django-15061',
                          'django__django-15320',
                          'django__django-15388',
                          'django__django-15400',
                          'django__django-15814',
                          'django__django-15851',
                          'django__django-15902',
                          'django__django-16046',
                          'django__django-16255',
                          'django__django-16379',
                          'django__django-16527',
                          'django__django-17087',
                          'matplotlib__matplotlib-23314',
                          'matplotlib__matplotlib-23476',
                          'matplotlib__matplotlib-23563',
                          'matplotlib__matplotlib-25433',
                          'mwaskom__seaborn-3010',
                          'mwaskom__seaborn-3190',
                          'mwaskom__seaborn-3407',
                          'psf__requests-1963',
                          'psf__requests-863',
                          'pydata__xarray-4094',
                          'pydata__xarray-4248',
                          'pydata__xarray-5131',
                          'pylint-dev__pylint-7080',
                          'pylint-dev__pylint-7993',
                          'pytest-dev__pytest-11143',
                          'pytest-dev__pytest-11148',
                          'pytest-dev__pytest-5227',
                          'pytest-dev__pytest-5413',
                          'pytest-dev__pytest-6116',
                          'pytest-dev__pytest-7168',
                          'pytest-dev__pytest-7432',
                          'scikit-learn__scikit-learn-13439',
                          'scikit-learn__scikit-learn-13584',
                          'scikit-learn__scikit-learn-13779',
                          'sphinx-doc__sphinx-7738',
                          'sphinx-doc__sphinx-8721',
                          'sympy__sympy-12171',
                          'sympy__sympy-13177',
                          'sympy__sympy-13480',
                          'sympy__sympy-13647',
                          'sympy__sympy-13971',
                          'sympy__sympy-14774',
                          'sympy__sympy-14817',
                          'sympy__sympy-15346',
                          'sympy__sympy-15609',
                          'sympy__sympy-16988',
                          'sympy__sympy-17139',
                          'sympy__sympy-17630',
                          'sympy__sympy-17655',
                          'sympy__sympy-18057',
                          'sympy__sympy-18621',
                          'sympy__sympy-19487',
                          'sympy__sympy-20212',
                          'sympy__sympy-21612',
                          'sympy__sympy-21614',
                          'sympy__sympy-21627',
                          'sympy__sympy-22005']}


In [7]:
# Custom-instance inputs inlined — no upstream to fetch.
CUSTOM_ROWS = []


In [8]:
output_dir = RUN_OUTPUT_DIR
output_path = output_dir / 'prediction-chain-9b.jsonl'
INSTANCES_PATH = output_dir / 'instances.jsonl'

selected = []
for (subset, split), ids in SWEBENCH_GROUPS.items():
    print(f'Fetching {len(ids)} {subset}/{split} row(s) from HuggingFace…')
    selected.extend(fetch_swebench_instances(subset, split, instance_ids=ids))
selected.extend(CUSTOM_ROWS)

with INSTANCES_PATH.open('w', encoding='utf-8') as _fh:
    for _row in selected:
        _fh.write(json.dumps(_row, ensure_ascii=False) + '\n')
print(f'Wrote {len(selected)} instance row(s) -> {INSTANCES_PATH}')

_have = {i['instance_id'] for i in selected}
missing = [iid for iid in INSTANCE_IDS if iid not in _have]
if missing:
    print('Missing rows (id not found in HF or in CUSTOM_ROWS):', missing)
print(f'Ready to run {len(selected)} instance(s).')


Fetching 77 lite/test row(s) from HuggingFace…


C:\Users\XF\.evomas-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-06-07 15:45:19,706 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/datasets/SWE-bench/SWE-bench_Lite/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"


2026-06-07 15:45:19,714 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/SWE-bench/SWE-bench_Lite/69611d31007e1c6731db8bd5b5c3f2d33f5bab6e/README.md "HTTP/1.1 200 OK"


2026-06-07 15:45:19,841 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/datasets/SWE-bench/SWE-bench_Lite/resolve/69611d31007e1c6731db8bd5b5c3f2d33f5bab6e/SWE-bench_Lite.py "HTTP/1.1 404 Not Found"


...

2026-06-07 15:45:20,697 [INFO] httpx: HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=SWE-bench/SWE-bench_Lite "HTTP/1.1 200 OK"


2026-06-07 15:45:20,828 [INFO] httpx: HTTP Request: GET https://huggingface.co/api/datasets/SWE-bench/SWE-bench_Lite/tree/69611d31007e1c6731db8bd5b5c3f2d33f5bab6e/data?recursive=true&expand=false "HTTP/1.1 200 OK"


2026-06-07 15:45:20,945 [INFO] httpx: HTTP Request: GET https://huggingface.co/api/datasets/SWE-bench/SWE-bench_Lite/tree/69611d31007e1c6731db8bd5b5c3f2d33f5bab6e?recursive=false&expand=false "HTTP/1.1 200 OK"


2026-06-07 15:45:21,073 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/datasets/SWE-bench/SWE-bench_Lite/resolve/69611d31007e1c6731db8bd5b5c3f2d33f5bab6e/dataset_infos.json "HTTP/1.1 404 Not Found"


Wrote 77 instance row(s) -> C:\Users\XF\Desktop\TFG\EvoMas\notebooks\notebook-chain-9b\instances.jsonl
Ready to run 77 instance(s).


## 4. Inference

Re-runs the EvoMas workflow for each instance with the inlined config. All notebook-produced artefacts (prediction JSONL, evaluation reports, custom-instance sidecar) land under one per-run folder at `notebook-chain-9b/` so they stay grouped together and don't mix with UI/CLI runs in the repo's `results/` tree.

In [9]:
from evomas.exceptions.errors import OllamaMemoryError

predictions = []
with open(output_path, 'w', encoding='utf-8') as out:
    for inst in selected:
        iid = inst['instance_id']
        print(f'--- {iid} ---')
        try:
            patch = run_evomas(inst, config=CONFIG)
        except OllamaMemoryError as exc:
            print(f'Ollama OOM; aborting: {exc}')
            break
        except Exception as exc:
            print(f'run failed on {iid}: {exc}')
            patch = ''
        rec = {
            'instance_id': iid,
            'model_patch': patch,
            'model_name_or_path': 'evomas-notebook',
        }
        predictions.append(rec)
        out.write(json.dumps(rec) + '\n')
print(f'Wrote {len(predictions)} prediction(s) to {output_path}.')


2026-06-07 15:45:21,183 [WARNING] weave.trace.op: Warning: Traces will not be logged. Call weave.init to log your traces to a project.
 (subsequent messages of this type will be suppressed)


2026-06-07 15:45:21,184 [INFO] evomas.core.workflow.runner: === running astropy__astropy-12907 with inline config (id=chain-9b) ===


--- astropy__astropy-12907 ---


2026-06-07 15:45:21,581 [INFO] evomas.utils.workspace: reusing workspace at C:\Users\XF\AppData\Local\Temp\evomas_workspace\astropy__astropy-12907 (HEAD=d16bfe05a744909de4b27f5875fe0d4ed41ce607)


2026-06-07 15:45:21,868 [INFO] evomas.core.workflow.runner: graph runtime: 4 agents x 2 max revisits => recursion_limit=8


...

2026-06-08 07:50:18,416 [INFO] evomas.models.langchain_ollama_model: [finalizer|resp ] patch accepted: the fix correctly raises NotImplementedError for infinite solution cases where the number of polynomials in the Gröbner basis is less than the number of variables.


2026-06-08 07:50:18,417 [INFO] evomas.models.langchain_ollama_model: [finalizer] tokens in=1717 out=115 total=1832


2026-06-08 07:50:18,417 [INFO] evomas.agents.finalizer: [finalizer] no tool calls — stopping loop


2026-06-08 07:50:18,906 [INFO] evomas.core.workflow.runner: === sympy__sympy-22005 done: 489-char patch | tokens in=99885 out=3621 total=103506 ===


Wrote 77 prediction(s) to C:\Users\XF\Desktop\TFG\EvoMas\notebooks\notebook-chain-9b\prediction-chain-9b.jsonl.


## 5. Evaluation

Runs `scripts/evaluation/run_swebench_evaluation.py` — the evaluator chosen at notebook-generation time (the `--evaluator` flag on `evomas notebook`). Forwards through the unified evaluator CLI contract: `--predictions / --instances / --report-dir / --run-id / --model`. Output lands at `<report-dir>/{model}.{run-id}.json` plus per-instance folders under `<report-dir>/logs/run_evaluation/<run-id>/<model>/<instance>/`.

This evaluator is marked `needs_wsl=True`, so on Windows the script is wrapped through WSL (the harness is POSIX-only).

In [10]:
EVALUATOR_STEM = 'run_swebench_evaluation'
EVALUATOR_NEEDS_WSL = True

first = selected[0] if selected else None
SUBSET = (first or {}).get('subset', 'lite')
SPLIT  = (first or {}).get('split',  'dev')

eval_report_dir = output_dir

import platform
from evomas.paths import BASE_DIR as _BASE_DIR
_script = _BASE_DIR / 'scripts' / 'evaluation' / f'{EVALUATOR_STEM}.py'
if EVALUATOR_NEEDS_WSL and platform.system() == 'Windows':
    from evomas.utils.paths import to_wsl
    cmd = [
        'wsl', '--', 'python3', to_wsl(str(_script)),
        '--predictions', to_wsl(str(output_path)),
        '--instances',   to_wsl(str(INSTANCES_PATH)),
        '--report-dir',  to_wsl(str(eval_report_dir)),
        '--run-id',      f'notebook-{SUBSET}-{SPLIT}',
        '--model',       'evomas-notebook',
    ]
else:
    cmd = [
        sys.executable, str(_script),
        '--predictions', str(output_path),
        '--instances',   str(INSTANCES_PATH),
        '--report-dir',  str(eval_report_dir),
        '--run-id',      f'notebook-{SUBSET}-{SPLIT}',
        '--model',       'evomas-notebook',
    ]
print(f'Evaluating via {EVALUATOR_STEM}.py')
print('+ ' + ' '.join(cmd))

eval_proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, encoding='utf-8', errors='replace',
)
assert eval_proc.stdout is not None
for line in eval_proc.stdout:
    print(line, end='')
eval_proc.wait()
print(f'\n[evaluation finished with exit code {eval_proc.returncode}]')

logs_root = eval_report_dir / 'logs' / 'run_evaluation'
if logs_root.is_dir():
    print('\nPer-instance artifacts:')
    for inst_dir in sorted(logs_root.rglob('*/')):
        if (inst_dir / 'report.json').is_file():
            print(f'  {inst_dir}')
for summary in sorted(eval_report_dir.glob('*.json')):
    print(f'Summary: {summary}')


Evaluating via run_swebench_evaluation.py
+ wsl -- python3 /mnt/c/Users/XF/Desktop/TFG/EvoMas/scripts/evaluation/run_swebench_evaluation.py --predictions /mnt/c/Users/XF/Desktop/TFG/EvoMas/notebooks/notebook-chain-9b/prediction-chain-9b.jsonl --instances /mnt/c/Users/XF/Desktop/TFG/EvoMas/notebooks/notebook-chain-9b/instances.jsonl --report-dir /mnt/c/Users/XF/Desktop/TFG/EvoMas/notebooks/notebook-chain-9b --run-id notebook-lite-test --model evomas-notebook


2026-06-08 07:50:19,142 - INFO - Bucket lite/test -> 77 row(s), run_id=notebook-lite-test
2026-06-08 07:50:19,203 - INFO - Running evaluation with 8 workers on /mnt/c/Users/XF/Desktop/TFG/EvoMas/notebooks/notebook-chain-9b/_tmp_predictions/_bucket_lite_test.jsonl (run_id=notebook-lite-test, dataset=SWE-bench/SWE-bench_Lite, split=test, force=True, cache_level=env, force_rebuild=False, python=/mnt/c/Users/XF/Desktop/TFG/EvoMas/SWE-bench/venv/bin/python, cwd=/mnt/c/Users/XF/Desktop/TFG/EvoMas/notebooks/notebook-chain-9b)


<frozen runpy>:128: RuntimeWarning: 'swebench.harness.run_evaluation' found in sys.modules after import of package 'swebench.harness', but prior to execution of 'swebench.harness.run_evaluation'; this may result in unpredictable behaviour
2026-06-08 07:50:38,036 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/SWE-bench/SWE-bench_Lite/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-08 07:50:38,044 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/SWE-bench/SWE-bench_Lite/69611d31007e1c6731db8bd5b5c3f2d33f5bab6e/README.md "HTTP/1.1 200 OK"


2026-06-08 07:50:38,161 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/SWE-bench/SWE-bench_Lite/resolve/69611d31007e1c6731db8bd5b5c3f2d33f5bab6e/SWE-bench_Lite.py "HTTP/1.1 404 Not Found"


2026-06-08 07:50:38,457 - httpx - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/SWE-bench/SWE-bench_Lite/SWE-bench/SWE-bench_Lite.py "HTTP/1.1 404 Not Found"
2026-06-08 07:50:38,588 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/datasets/SWE-bench/SWE-bench_Lite/revision/69611d31007e1c6731db8bd5b5c3f2d33f5bab6e "HTTP/1.1 200 OK"


...

2026-06-08 07:50:39,523 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/SWE-bench/SWE-bench_Lite/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-08 07:50:39,531 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/SWE-bench/SWE-bench_Lite/69611d31007e1c6731db8bd5b5c3f2d33f5bab6e/README.md "HTTP/1.1 200 OK"
2026-06-08 07:50:39,649 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/SWE-bench/SWE-bench_Lite/resolve/69611d31007e1c6731db8bd5b5c3f2d33f5bab6e/SWE-bench_Lite.py "HTTP/1.1 404 Not Found"


2026-06-08 07:50:39,744 - httpx - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/SWE-bench/SWE-bench_Lite/SWE-bench/SWE-bench_Lite.py "HTTP/1.1 404 Not Found"
2026-06-08 07:50:39,865 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/SWE-bench/SWE-bench_Lite/resolve/69611d31007e1c6731db8bd5b5c3f2d33f5bab6e/.huggingface.yaml "HTTP/1.1 404 Not Found"
2026-06-08 07:50:39,866 - huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


2026-06-08 07:50:40,005 - httpx - INFO - HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=SWE-bench/SWE-bench_Lite "HTTP/1.1 200 OK"
2026-06-08 07:50:40,125 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/SWE-bench/SWE-bench_Lite/resolve/69611d31007e1c6731db8bd5b5c3f2d33f5bab6e/dataset_infos.json "HTTP/1.1 404 Not Found"
Traceback (most recent call last):
  File "/mnt/c/Users/XF/Desktop/TFG/EvoMas/SWE-bench/venv/lib/python3.12/site-packages/urllib3/connectionpool.py", line 787, in urlopen
    response = self._make_request(
               ^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/XF/Desktop/TFG/EvoMas/SWE-bench/venv/lib/python3.12/site-packages/urllib3/connectionpool.py", line 493, in _make_request
    conn.request(
  File "/mnt/c/Users/XF/Desktop/TFG/EvoMas/SWE-bench/venv/lib/python3.12/site-packages/urllib3/connection.py", line 500, in request
    self.endheaders()
  File "/usr/lib/python3.12/http/client.py", line 1351, in endheaders
    self._s

    sock.connect(self.unix_socket)
FileNotFoundError: [Errno 2] No such file or directory

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/mnt/c/Users/XF/Desktop/TFG/EvoMas/SWE-bench/venv/lib/python3.12/site-packages/requests/adapters.py", line 645, in send
    resp = conn.urlopen(
           ^^^^^^^^^^^^^
  File "/mnt/c/Users/XF/Desktop/TFG/EvoMas/SWE-bench/venv/lib/python3.12/site-packages/urllib3/connectionpool.py", line 841, in urlopen
    retries = retries.increment(
              ^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/XF/Desktop/TFG/EvoMas/SWE-bench/venv/lib/python3.12/site-packages/urllib3/util/retry.py", line 490, in increment
    raise reraise(type(error), error, _stacktrace)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/XF/Desktop/TFG/EvoMas/SWE-bench/venv/lib/python3.12/site-packages/urllib3/util/util.py", line 38, in reraise
    raise value.with_traceback(tb)
  File "/mnt/c/Users/XF/


[evaluation finished with exit code 1]
